# 00 · Data Pipeline — Luồng Thu Thập & Xử Lý Dữ Liệu

Notebook này **mô phỏng** quy trình data pipeline bằng cách đọc lại các file đã được xử lý trong `data/processed/`.

Để chạy lại pipeline thật (gọi API thực tế, scrape tin tức):
- Chạy `python scripts/fetch_data.py` (thu thập giá OHLCV và tính indicators)
- Chạy `python scripts/fetch_news.py` (thu thập tin tức vnstock + CafeF sitemap và align)

**Mục tiêu**:
1. Trình bày trực quan luồng dữ liệu phục vụ buổi báo cáo/defense.
2. Validate schema đầu ra của giá (OHLCV + indicators) và tin tức.
3. Kiểm tra tính đúng đắn của lookahead-safe invariant trên tập dữ liệu đã thu thập.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Make sharing modules work
_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
_PROJECT_ROOT = _NB_DIR.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src import config
from src.data_pipeline.indicators import INDICATOR_COLS
from src.data_pipeline.calendar import build_trading_calendar
from src.data_pipeline.news_align import NEWS_SCHEMA, visible_news_at

DATA = _PROJECT_ROOT / "data" / "processed"
TICKERS = config.TICKERS
print("Setup OK · tickers:", TICKERS)

Setup OK · tickers: ['VCB', 'FPT', 'HPG', 'VIC', 'VNM']


## Bước 1 — Thu thập giá OHLCV từ vnstock

**Trong thực tế** (xử lý qua `scripts/fetch_data.py` gọi hàm trong `src/data_pipeline/vnstock_prices.py`):
```python
# Với mỗi ticker trong vũ trụ VN30 (TICKERS):
from src.data_pipeline.vnstock_prices import fetch_prices
df = fetch_prices("VCB", start="2019-01-01", end="2026-04-30")
# Trả về dataframe định dạng chuẩn: [date, ticker, open, high, low, close, volume]
```

Ở bước này chúng ta sẽ đọc dữ liệu đã xử lý để minh họa cấu trúc dữ liệu thô thu thập được.

In [2]:
# Đọc dữ liệu prices đã lưu từ data/processed/
prices = pd.read_parquet(DATA / "prices.parquet")
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()

BASE_COLS = ["date", "ticker", "open", "high", "low", "close", "volume"]
assert set(TICKERS) == set(prices["ticker"].unique())
assert set(BASE_COLS).issubset(prices.columns)

print("=== OHLCV Schema (output của fetch_prices) ===")
print(prices[BASE_COLS].head(5).to_string(index=False))
print(f"\nSố phiên giao dịch thô mỗi ticker:")
print(prices.groupby("ticker").size().to_string())
print(f"\nKhoảng thời gian dữ liệu giá: {prices['date'].min().date()} → {prices['date'].max().date()}")

=== OHLCV Schema (output của fetch_prices) ===
      date ticker  open  high   low  close  volume
2019-01-02    VCB 23.09 23.26 22.92  22.96 1081640
2019-01-03    VCB 23.13 23.17 22.57  22.96 1071350
2019-01-04    VCB 22.83 23.30 22.53  23.30 1307310
2019-01-07    VCB 23.69 23.77 23.34  23.39 1175810
2019-01-08    VCB 23.56 23.64 23.34  23.56 1318810

Số phiên giao dịch thô mỗi ticker:
ticker
FPT    1826
HPG    1826
VCB    1826
VIC    1826
VNM    1826

Khoảng thời gian dữ liệu giá: 2019-01-02 → 2026-04-29


## Bước 1b — Tính toán Technical Indicators (Chỉ báo kỹ thuật)

**Trong thực tế** (xử lý qua `src/data_pipeline/indicators.py`):
```python
from src.data_pipeline.indicators import apply_indicators
prices_with_indicators = apply_indicators(prices_raw)
# Áp dụng các chỉ báo kỹ thuật: RSI-14, MACD, SMA(5, 20, 50), Bollinger Bands (20, 2), ATR-14
```
⚠️ **Nguyên tắc cô lập**: Chỉ báo được tính độc lập cho từng mã cổ phiếu (không sử dụng thông tin của mã khác hoặc tính toán xuyên mã) để đảm bảo không rò rỉ dữ liệu chéo (cross-ticker leakage).

In [3]:
# Kiểm tra các chỉ báo kỹ thuật trong tập dữ liệu
assert set(INDICATOR_COLS).issubset(prices.columns), \
    f"Thiếu chỉ báo: {set(INDICATOR_COLS) - set(prices.columns)}"

print("=== Technical Indicators Computed per Ticker ===")
print(f"Danh sách các chỉ báo kỹ thuật: {list(INDICATOR_COLS)}")
print()

# Minh hoạ 5 hàng cuối của VCB cùng các chỉ báo đã tính
vcb = prices[prices["ticker"] == "VCB"].sort_values("date").tail(5)
display_cols = ["date", "close", "rsi14", "macd", "sma20", "bb_upper", "bb_lower", "atr14"]
print(vcb[display_cols].to_string(index=False))

=== Technical Indicators Computed per Ticker ===
Danh sách các chỉ báo kỹ thuật: ['rsi14', 'macd', 'macd_signal', 'sma5', 'sma20', 'sma50', 'bb_upper', 'bb_lower', 'atr14']

      date  close     rsi14      macd  sma20  bb_upper  bb_lower    atr14
2026-04-22   59.4 48.250385 -0.232973 58.975 60.568581 57.381419 1.139711
2026-04-23   62.8 63.388655  0.054418 59.220 61.454815 56.985185 1.351160
2026-04-24   60.6 52.655214  0.103462 59.310 61.613823 57.006177 1.426077
2026-04-28   59.8 49.380772  0.076891 59.400 61.631591 57.168409 1.445643
2026-04-29   59.8 49.380772  0.055196 59.485 61.640249 57.329751 1.413812


## Bước 2 — Xây dựng Trading Calendar (Lịch giao dịch)

**Trong thực tế** (xử lý qua `src/data_pipeline/calendar.py`):
```python
from src.data_pipeline.calendar import build_trading_calendar
calendar = build_trading_calendar(prices)
```
Thay vì dùng các thư viện lịch giao dịch bên thứ ba (không phản ánh chính xác ngày nghỉ lễ ở Việt Nam), lịch giao dịch được xây dựng trực tiếp từ hợp của các ngày giao dịch thực tế quan sát được trong dữ liệu giá (Empirical Trading Calendar).

In [4]:
# Xây dựng lịch giao dịch thực nghiệm từ dữ liệu
calendar = build_trading_calendar(prices)
print(f"Tổng số phiên giao dịch: {len(calendar)} phiên")
print(f"Phiên giao dịch đầu tiên: {calendar[0].date()}")
print(f"Phiên giao dịch cuối cùng:  {calendar[-1].date()}")

# Tìm các khoảng trống (gaps) lớn để phát hiện kỳ nghỉ lễ lớn
gaps = pd.Series(calendar, name="date").diff().dt.days.dropna()
long_gaps = gaps[gaps > 3].sort_values(ascending=False).head(5)
print(f"\nTop 5 khoảng trống dài nhất giữa các phiên (nghỉ Tết/lễ dài ngày):")
for idx, g in long_gaps.items():
    start_gap = calendar[idx - 1].date()
    end_gap = calendar[idx].date()
    print(f"  Từ {start_gap} đến {end_gap}: khoảng trống {int(g)} ngày")

Tổng số phiên giao dịch: 1826 phiên
Phiên giao dịch đầu tiên: 2019-01-02
Phiên giao dịch cuối cùng:  2026-04-29

Top 5 khoảng trống dài nhất giữa các phiên (nghỉ Tết/lễ dài ngày):
  Từ 2019-02-01 đến 2019-02-11: khoảng trống 10 ngày
  Từ 2026-02-13 đến 2026-02-23: khoảng trống 10 ngày
  Từ 2025-01-24 đến 2025-02-03: khoảng trống 10 ngày
  Từ 2022-01-28 đến 2022-02-07: khoảng trống 10 ngày
  Từ 2020-01-22 đến 2020-01-30: khoảng trống 8 ngày


## Bước 3 — Thu thập tin tức và căn chỉnh thời gian khả dụng (Align News)

**Trong thực tế** (xử lý qua `scripts/fetch_news.py`):
Quy trình bao gồm 2 nguồn tin chính:
1. **vnstock Company.news()** (VCI source): Thu thập ~50 tin tức công bố thông tin gần nhất của từng mã cổ phiếu.
2. **CafeF XML sitemaps**: Quét XML sitemap theo khoảng thời gian chỉ định (`scrape_cafef_sitemap_range`) để lấy thông tin các bài báo, chuẩn hóa chữ tiếng Việt không dấu rồi dùng regex khớp mã cổ phiếu nhằm gán tag mã cổ phiếu (`tag_tickers`).

Sau khi gộp và loại bỏ trùng lặp dựa trên URL (giữ lại các bản tin không có URL như công bố thông tin nội bộ của vnstock), trường `available_for_session` sẽ được tính toán dựa trên lịch giao dịch để phục vụ luật **Không có Lookahead Bias**.

In [5]:
# Đọc dữ liệu tin tức đã xử lý
news = pd.read_parquet(DATA / "news.parquet")
news["published_at_utc"] = pd.to_datetime(news["published_at_utc"], utc=True)
news["available_for_session"] = pd.to_datetime(news["available_for_session"])

assert list(news.columns) == NEWS_SCHEMA

print("=== Thống kê dữ liệu tin tức ===")
print(f"Tổng số tin thu thập được: {len(news)} tin")
print(f"\nPhân bố theo nguồn tin:")
print(news["source"].value_counts().to_string())
print(f"\nSố tin tức có thể sử dụng (đã căn chỉnh session): {news['available_for_session'].notna().sum()}")
print(f"\nMột số tin tức mẫu:")
print(news[["published_at_utc", "available_for_session", "source", "title"]].head(3).to_string(index=False))

=== Thống kê dữ liệu tin tức ===
Tổng số tin thu thập được: 2039 tin

Phân bố theo nguồn tin:
source
cafef      1789
vnstock     250

Số tin tức có thể sử dụng (đã căn chỉnh session): 1992

Một số tin tức mẫu:
         published_at_utc available_for_session source                                                                                                              title
2025-05-01 03:30:33+00:00            2025-05-06  cafef <![CDATA[VPBank, Vietcombank, Techcombank thông báo tạm dừng giao dịch chuyển tiền, rút tiền với trường hợp sau]]>
2025-05-01 03:37:38+00:00            2025-05-06  cafef                                <![CDATA[Kỷ lục 84.000 tỷ doanh thu của Vingroup: 3,2 tỷ USD đổ về từ những đâu?]]>
2025-05-02 03:58:53+00:00            2025-05-06  cafef                                          <![CDATA[Hòa Phát nợ vay gần 90.000 tỷ, tiền mặt xuống thấp nhất 4 năm]]>


## Bước 3b — Cơ chế Căn chỉnh Tránh Lookahead Bias

**Quy tắc Lookahead-Safe (CLAUDE.md §1 & PRD §11)**:
- Tin tức xuất hiện vào ngày D (giờ địa phương).
- Tin này chỉ được phép có tác động vào quyết định giao dịch tại đầu phiên của ngày **D+2** (hoặc nói cách khác, có hiệu lực từ CLOSE ngày D+1).
- `available_for_session` được gán bằng phiên giao dịch thứ hai tính từ ngày xuất hiện tin.

Consumer quyết định tại phiên giao dịch T sẽ chỉ được phép xem các tin có `available_for_session <= T`.

In [6]:
# Chọn một phiên giao dịch giả định
SAMPLE_SESSION = "2025-06-10"
asof = pd.Timestamp(SAMPLE_SESSION)

# Lấy các tin tức khả dụng tại đầu phiên giả định
visible = visible_news_at(news, asof_session=SAMPLE_SESSION)

print(f"=== Các tin tức có thể sử dụng tại đầu phiên {SAMPLE_SESSION} ===")
print(f"Số lượng tin khả dụng: {len(visible)} tin")
print()

# Kiểm tra tính đúng đắn
assert (visible["available_for_session"] <= asof).all(), "PHÁT HIỆN LOOKAHEAD BIAS!"
print("✅ Xác thực: Không có tin tức nào có session hiệu lực lớn hơn session hiện tại. Lookahead-safe OK.")
print()

# Hiển thị 3 tin gần nhất khả dụng tại thời điểm này
print("3 tin tức mới nhất có hiệu lực:")
recent = visible.sort_values("available_for_session", ascending=False).head(3)
print(recent[["published_at_utc", "available_for_session", "source", "title"]].to_string(index=False))

=== Các tin tức có thể sử dụng tại đầu phiên 2025-06-10 ===
Số lượng tin khả dụng: 169 tin

✅ Xác thực: Không có tin tức nào có session hiệu lực lớn hơn session hiện tại. Lookahead-safe OK.

3 tin tức mới nhất có hiệu lực:
         published_at_utc available_for_session source                                                                                                               title
2025-06-07 10:08:20+00:00            2025-06-10  cafef                                 <![CDATA[Vingroup bắt tay với một sàn TMĐT phát triển hạ tầng giao nhận hàng hoá]]>
2025-06-06 15:58:01+00:00            2025-06-10  cafef                                      <![CDATA[Thị giá cao ngất ngưởng, FPT Retail chuẩn bị chia cổ tức tỷ lệ 25%]]>
2025-06-06 09:31:34+00:00            2025-06-10  cafef <![CDATA[Động thái quyết liệt của tỉnh Quảng Ngãi tại dự án sản xuất ray đường sắt và thép đặc biệt của Hòa Phát]]>


In [8]:
# Final summary
print("=" * 60)
print("  DATA PIPELINE — TÓM TẮT LUỒNG")
print("=" * 60)
print("""
vnstock API (KBS→VCI fallback)
    ↓  fetch_prices() × 5 tickers
    ↓  apply_indicators()  (per-ticker, no cross-leakage)
    └──→ prices.parquet  [date, ticker, OHLCV, RSI, MACD, SMA, BB, ATR]

trading calendar  ←  derived from prices dates (empirical HOSE)
    │
    ├── vnstock Company.news()  × 5 tickers  (~50 tin/ticker)
    │
    └── CafeF sitemap XML  (date range, keyword-match alias)
            ↓  merge + dedup by URL
            ↓  compute available_for_session  (D → D+2 open)
            └──→ news.parquet  [published_at_utc, available_for_session, ...]
""")
print(f"prices.parquet : {len(prices):,} rows × {prices.shape[1]} cols")
print(f"news.parquet   : {len(news):,} rows × {news.shape[1]} cols")
print(f"calendar       : {len(calendar):,} trading sessions")


  DATA PIPELINE — TÓM TẮT LUỒNG

vnstock API (KBS→VCI fallback)
    ↓  fetch_prices() × 5 tickers
    ↓  apply_indicators()  (per-ticker, no cross-leakage)
    └──→ prices.parquet  [date, ticker, OHLCV, RSI, MACD, SMA, BB, ATR]

trading calendar  ←  derived from prices dates (empirical HOSE)
    │
    ├── vnstock Company.news()  × 5 tickers  (~50 tin/ticker)
    │
    └── CafeF sitemap XML  (date range, keyword-match alias)
            ↓  merge + dedup by URL
            ↓  compute available_for_session  (D → D+2 open)
            └──→ news.parquet  [published_at_utc, available_for_session, ...]

prices.parquet : 9,130 rows × 16 cols
news.parquet   : 2,039 rows × 7 cols
calendar       : 1,826 trading sessions


In [ ]:
# %% Defense Q&A
# Q1: Tại sao dùng vnstock thay vì Alpha Vantage hay Yahoo Finance?
#   A: Yahoo Finance không cung cấp đầy đủ thông tin chi tiết về sàn HOSE và tin tức đi kèm.
#      vnstock là thư viện chính quy cho thị trường Việt Nam, truy xuất trực tiếp nguồn dữ liệu từ các công ty chứng khoán lớn.
#
# Q2: Tại sao phải sử dụng đồng thời 2 nguồn tin (vnstock và CafeF sitemap)?
#   A: vnstock Company.news API miễn phí bị giới hạn tối đa 50 tin cho mỗi mã (chỉ cover 8-10 tháng gần nhất).
#      CafeF sitemap scraper cho phép quét sitemap quá khứ theo ngày để bao phủ toàn bộ giai đoạn test 12 tháng.
#
# Q3: Thuật toán căn chỉnh available_for_session hoạt động thế nào?
#   A: Tin tức giờ UTC -> Chuyển múi giờ Việt Nam (UTC+7) -> Xác định ngày xuất bản D.
#      Tìm ngày giao dịch đầu tiên sau ngày D trên Lịch giao dịch -> Đó là phiên D+1.
#      Lấy phiên giao dịch tiếp theo kế tiếp phiên D+1 -> Phiên D+2. Đây là phiên đầu tiên tác nhân giao dịch được dùng tin này.
#
# Q4: Tại sao không dùng các thư viện lịch nghỉ lễ tiêu chuẩn?
#   A: Các thư viện lịch quốc tế thường cập nhật ngày nghỉ lễ Việt Nam không chính xác hoặc chậm.
#      Việc tự xây dựng Lịch giao dịch từ union của dữ liệu giá thực tế là cách chính xác nhất và phản ánh đúng thực tế vận hành.
#
# Q5: Dữ liệu indicators có bị rò rỉ chéo giữa các mã không?
#   A: Không. indicators.py được viết tách biệt hoàn toàn qua một vòng lặp groupby('ticker') để tính toán độc lập cho từng ticker.
#      Sau đó mới ghép (pd.concat) trở lại định dạng long format.